In [6]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import gc
import os

# CLEANUP

gc.collect()
tf.keras.backend.clear_session()

# CONFIG

TRAIN_PATH = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
TEST_PATH  = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'
IMG_SIZE   = (224, 224)  
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE
all_images = []
all_labels = []
class_names = sorted(os.listdir(TRAIN_PATH))
NUM_CLASSES = len(class_names)
print("Classes:", class_names)

for label_idx, class_name in enumerate(class_names):
    class_dir = os.path.join(TRAIN_PATH, class_name)
    for fname in os.listdir(class_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append(os.path.join(class_dir, fname))
            all_labels.append(label_idx)


# TRAIN / VALIDATION SPLIT

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_images, all_labels,
    test_size=0.15,
    stratify=all_labels,
    random_state=42
)
print(f"Train: {len(train_paths)}")
print(f"Validation: {len(val_paths)}")
print("Validation Distribution:"),
      {class_names[k]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)


# IMAGE LOADING 

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

# AUGMENTATION 

_aug_rotate = tf.keras.layers.RandomRotation(0.10, fill_mode='reflect')
_aug_zoom   = tf.keras.layers.RandomZoom(0.10,   fill_mode='reflect')

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)

    img = tf.expand_dims(img, 0)          
    img = _aug_rotate(img, training=True)
    img = _aug_zoom(img,   training=True)
    img = tf.squeeze(img, 0)              

    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

# TRAIN DATASET

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds = train_ds.shuffle(len(train_paths), seed=42, reshuffle_each_iteration=True)
train_ds = train_ds.map(load_image, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.cache()                        
train_ds = train_ds.map(augment, num_parallel_calls=AUTOTUNE)  
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(AUTOTUNE)


# VALIDATION DATASET

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(load_image, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.cache()
val_ds = val_ds.batch(BATCH_SIZE)
val_ds = val_ds.prefetch(AUTOTUNE)

# TEST DATASET

test_images, test_labels = [], []
for label_idx, class_name in enumerate(class_names):
    class_dir = os.path.join(TEST_PATH, class_name)
    for fname in os.listdir(class_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            test_images.append(os.path.join(class_dir, fname))
            test_labels.append(label_idx)

test_ds = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
test_ds = test_ds.map(load_image, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.cache()
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(AUTOTUNE)

# MODEL 


def residual_block(x, filters, stride=1):
    """
    Pre-activation residual block (BN → ReLU → Conv).
    Shortcut uses 1×1 conv when dims change.
    """
    shortcut = x

    # Main path
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(
        filters, 3, strides=stride, padding='same',
        use_bias=False, kernel_regularizer=l2(1e-4)
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(
        filters, 3, padding='same',
        use_bias=False, kernel_regularizer=l2(1e-4)
    )(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters, 1, strides=stride, padding='same',
            use_bias=False, kernel_regularizer=l2(1e-4)
        )(shortcut)

    x = layers.Add()([x, shortcut])
    return x


def build_model():
    inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

    # ── Stem ──────────────────────────────────────────
    x = layers.Conv2D(
        32, 7, strides=2, padding='same',
        use_bias=False, kernel_regularizer=l2(1e-4)
    )(inputs)                                          # 112×112×32
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same')(x)  # 56×56×32

    # ── Stage 1: 56×56 ────────────────────────────────
    x = residual_block(x, 64)
    x = residual_block(x, 64)

    # ── Stage 2: 28×28 ────────────────────────────────
    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)

    # ── Stage 3: 14×14 ────────────────────────────────
    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256)

    # ── Stage 4: 7×7 ──────────────────────────────────
    x = residual_block(x, 512, stride=2)
    x = residual_block(x, 512)

    # ── Head ──────────────────────────────────────────
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)                     
    x = layers.Dense(
        256, activation='relu',
        kernel_regularizer=l2(1e-4)
    )(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    return models.Model(inputs, outputs)

# COMPILE
model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    ),
    metrics=['accuracy']
}

def to_one_hot(img, label):
    return img, tf.one_hot(label, NUM_CLASSES)

train_ds_oh = train_ds.map(to_one_hot)
val_ds_oh   = val_ds.map(to_one_hot)
test_ds_oh  = test_ds.map(to_one_hot)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=3e-4,
        weight_decay=1e-4
    ),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

model.summary()

# CALLBACKS

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,            
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/kaggle/working/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# TRAIN

history = model.fit(
    train_ds_oh,
    validation_data=val_ds_oh,
    epochs=60,                  
    callbacks=callbacks,
    class_weight=class_weights
)

# EVALUATE

loss, acc = model.evaluate(test_ds_oh)
print(f"\nTest Accuracy : {round(acc, 4)}")
print(f"Test Loss     : {round(loss, 4)}")

# TRAINING CURVES

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Acc')
axes[0].plot(history.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()

# SAVE

model.save('/kaggle/working/best_model.keras')
print("Model saved successfully.")


IndentationError: unexpected indent (1236350555.py, line 49)

**Confusion Matrix**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_pred_probs = model.predict(test_ds_oh)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds_oh], axis=0)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names,
            yticklabels=class_names,
            cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

ModuleNotFoundError: No module named 'sklearn'

**F1 Score, Precision, Recall**

In [ ]:
# F1 SCORE, PRECISION, RECALL

from sklearn.metrics import classification_report

y_pred_probs = model.predict(test_ds_oh)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds_oh], axis=0)

print("\n========= Classification Report =========")
print(classification_report(y_true, y_pred, target_names=class_names))